In [1]:
!pip install -U langchain langchain-community langchain-core langchain-qdrant qdrant-client langchain-huggingface flashrank
!pip install fastembed sentence-transformers huggingface-hub>=0.36.2 
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!pip install -qU evaluate rouge_score bert_score transformers tqdm datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 88.3 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
 

In [2]:
import os
import shutil
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# Đăng nhập Hugging Face
user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("Hugging_face_access"))

# Mở khóa Qdrant DB
SOURCE_PATH = "/kaggle/input/datasets/haivuu/qdrant-v4" 
DB_PATH = "/kaggle/working/qdrant_writable_db"
DATA_PATH = "/kaggle/input/datasets/haivuu/llm-eval/legal_dataset_val.jsonl"
print("Đang chuẩn bị cơ sở dữ liệu...")
if not os.path.exists(DB_PATH):
    shutil.copytree(SOURCE_PATH, DB_PATH)
    print("Đã copy DB sang /working/")
else:
    print("DB đã có sẵn trong /working/")
    
lock_file = os.path.join(DB_PATH, ".lock")
if os.path.exists(lock_file):
    try: 
        os.remove(lock_file)
        print("Đã mở khóa DB thành công!")
    except: 
        pass

Đang chuẩn bị cơ sở dữ liệu...
Đã copy DB sang /working/
Đã mở khóa DB thành công!


In [3]:
import json
from datasets import Dataset

# 1. Giữ nguyên hàm load data an toàn của bạn
def load_jsonl_safe(path, max_samples=None):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                item = json.loads(line)
                if "question" in item and "answer" in item:
                    data.append(item)
                if max_samples and len(data) >= max_samples:
                    break
            except json.JSONDecodeError:
                continue
    return data

raw_data = load_jsonl_safe(DATA_PATH)
print(f" Đã đọc thành công {len(raw_data)} dòng dữ liệu thô!")

dataset = Dataset.from_list(raw_data)



eval_dataset = dataset
questions = eval_dataset["question"]
reference_answers = eval_dataset["answer"]

print(f" Đã chuẩn bị xong {len(questions)} câu hỏi Test (Hoàn toàn nguyên bản, chưa qua template)!")

 Đã đọc thành công 50 dòng dữ liệu thô!
 Đã chuẩn bị xong 50 câu hỏi Test (Hoàn toàn nguyên bản, chưa qua template)!


In [4]:
import ctypes
import llama_cpp
from huggingface_hub import hf_hub_download
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import LlamaCpp
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
# 1. Tải Embeddings
dense_embeddings = HuggingFaceEmbeddings(
    model_name="bkai-foundation-models/vietnamese-bi-encoder",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

# 2. Tải Llama.cpp (Tắt log cho đỡ rối mắt)
@llama_cpp.llama_log_callback
def null_log_callback(level, text, user_data): pass
llama_cpp.llama_log_set(null_log_callback, ctypes.c_void_p(0))

MODEL_PATH = hf_hub_download(repo_id="HaiVuuu/Vistral-7B-Chat-gguf", filename="Vistral-7B-Chat.Q4_K_M.gguf")
llm = LlamaCpp(
    model_path=MODEL_PATH, n_gpu_layers=-1, n_batch=1024, n_ctx=6096
    , f16_kv=True,temperature=0.0, top_p=0.9, top_k=40, repeat_penalty=1.05, max_tokens=1024, stop=["<|im_end|>"]
) 
# 3. Kết nối Qdrant
client = QdrantClient(path=DB_PATH)
qdrant_db = QdrantVectorStore(
    client=client, collection_name="legal_enriched_chunks", 
    embedding=dense_embeddings, sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID, vector_name="dense", sparse_vector_name="sparse"
)
SO_LUONG_BOC_THO = 15   
SO_LUONG_RERANK = 4  
base_retriever = qdrant_db.as_retriever(search_kwargs={"k": 10})
model_name = "BAAI/bge-reranker-v2-m3"
cross_encoder = HuggingFaceCrossEncoder(
    model_name=model_name, 
    model_kwargs={'device': 'cuda'}
)
compressor = CrossEncoderReranker(model=cross_encoder, top_n=SO_LUONG_RERANK)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# 4. Khởi tạo Chains
rewrite_template = """<|im_start|>system\nBạn là chuyên gia pháp lý. Hãy dịch câu hỏi sang ngôn ngữ pháp lý chuẩn xác.\nCHỈ TRẢ VỀ CÂU HỎI ĐÃ ĐƯỢC VIẾT LẠI.<|im_end|>\n<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"""
rewrite_chain = PromptTemplate.from_template(rewrite_template) | llm | StrOutputParser()

template = """<|im_start|>system\nBạn là Trợ lý Pháp lý AI. Chỉ dùng thông tin trong TÀI LIỆU để trả lời CÂU HỎI.\nBước 1 - Đánh giá: Trích xuất dữ kiện.\nBước 2 - Kết luận: Trả lời chi tiết bắt đầu bằng 'Căn cứ theo...'<|im_end|>\n<|im_start|>user\n[TÀI LIỆU]:\n{context}\n\n[CÂU HỎI]: {input}\n<|im_end|>\n<|im_start|>assistant\nBước 1 - Đánh giá:"""
document_prompt = PromptTemplate(input_variables=["page_content", "ten_van_ban"], template="[Trích từ: {ten_van_ban}]\n{page_content}")
document_chain = create_stuff_documents_chain(llm=llm, prompt=PromptTemplate.from_template(template), document_prompt=document_prompt)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [5]:
import torch
import evaluate
import numpy as np
from tqdm import tqdm

# Khởi tạo các metric từ HuggingFace
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')
perplexity = evaluate.load("perplexity", module_type="metric")

In [6]:
import gc
active_retriever = compression_retriever
def ask_ultimate_legal_bot_for_eval(question):
    rewritten_query = rewrite_chain.invoke({"question": question}).strip()
    source_docs = active_retriever.invoke(rewritten_query)
    source_docs = source_docs[:2]
    # Ở hàm chain cũ bạn dùng stream, nhưng ở đây invoke thẳng lấy chuỗi cho nhanh
    response = document_chain.invoke({"input": question, "context": source_docs})
    
    # Loại bỏ thẻ kết thúc nếu có
    if "<|im_end|>" in response:
        response = response.split("<|im_end|>")[0]
        
    # Thêm lại chữ 'Bước 1 - Đánh giá:' bị cắt mất do prompt
    return "Bước 1 - Đánh giá:" + response.strip()

print(f"Bắt đầu chạy đánh giá trên {len(eval_dataset)} test cases...")

# questions = [item["question"] for item in test_dataset]
# reference_answers = [item["ground_truth_gov"] for item in test_dataset]
generated_answers = []

for q in tqdm(questions, desc="Sinh câu trả lời AI"):
    ans = ask_ultimate_legal_bot_for_eval(q)
    generated_answers.append(ans if ans.strip() else "Không có câu trả lời.")

try:
    del llm
    del dense_embeddings
    del active_retriever
    del document_chain
except NameError:
    pass

# Ép Python dọn rác và làm sạch bộ nhớ đệm của PyTorch
gc.collect()
torch.cuda.empty_cache()

print("\nĐang tính toán các chỉ số...")
rouge_results = rouge.compute(predictions=generated_answers, references=reference_answers)
bert_results = bertscore.compute(predictions=generated_answers, references=reference_answers, lang="vi",batch_size=4)
ppl_results = perplexity.compute(predictions=generated_answers, model_id="Qwen/Qwen2.5-0.5B",batch_size=1)

print("\n" + "="*60)
print(" BÁO CÁO HIỆU NĂNG RAG")
print("="*60)
print(f"ROUGE-1: {round(rouge_results['rouge1'], 4)}")
print(f"ROUGE-2: {round(rouge_results['rouge2'], 4)}")
print(f"ROUGE-L: {round(rouge_results['rougeL'], 4)}")
print(f"BERTScore (F1 Trung bình): {round(np.mean(bert_results['f1']), 4)}")
print(f"Perplexity Trung bình: {round(ppl_results['mean_perplexity'], 4)}")
print("="*60)

Bắt đầu chạy đánh giá trên 50 test cases...


Sinh câu trả lời AI:   2%|▏         | 1/50 [00:35<28:55, 35.41s/it]Llama.generate: 12 prefix-match hit, remaining 118 prompt tokens to eval
Llama.generate: 12 prefix-match hit, remaining 1770 prompt tokens to eval
Sinh câu trả lời AI:   4%|▍         | 2/50 [01:12<29:05, 36.37s/it]Llama.generate: 12 prefix-match hit, remaining 110 prompt tokens to eval
Llama.generate: 12 prefix-match hit, remaining 476 prompt tokens to eval
Sinh câu trả lời AI:   6%|▌         | 3/50 [01:37<24:34, 31.36s/it]Llama.generate: 12 prefix-match hit, remaining 114 prompt tokens to eval
Llama.generate: 12 prefix-match hit, remaining 667 prompt tokens to eval
Sinh câu trả lời AI:   8%|▊         | 4/50 [02:04<22:34, 29.44s/it]Llama.generate: 12 prefix-match hit, remaining 106 prompt tokens to eval
Llama.generate: 12 prefix-match hit, remaining 718 prompt tokens to eval
Sinh câu trả lời AI:  10%|█         | 5/50 [02:28<20:32, 27.39s/it]Llama.generate: 12 prefix-match hit, remaining 103 prompt tokens to eval
Llama.g


Đang tính toán các chỉ số...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  0%|          | 0/50 [00:00<?, ?it/s]


 BÁO CÁO HIỆU NĂNG RAG
ROUGE-1: 0.5932
ROUGE-2: 0.354
ROUGE-L: 0.3318
BERTScore (F1 Trung bình): 0.7251
Perplexity Trung bình: 8.8719
